# P4-3 · Bài 6 · Đánh giá hệ thống AI

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

## Bài tập 5: Bộ chạy eval {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import json, statistics, time

def run_eval(path, search, answer, judge, k=5, out="eval_results.jsonl"):
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    results = []
    for r in rows:
        t0 = time.time()
        hits = search(r["question"], k)
        ans, _ = answer(r["question"])
        verdict = judge(r["question"], ans, r["reference"])           # {"reason", "score"}
        results.append({**r, "recall": len(set(hits) & set(r["gold_ids"])) / max(1, len(r["gold_ids"])),
                        "answer": ans, **verdict, "latency": time.time() - t0})
    with open(out, "w", encoding="utf-8") as f:
        for x in results:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    print(f"recall@{k} {statistics.mean(x['recall'] for x in results):.3f} | "
          f"điểm TB {statistics.mean(x['score'] for x in results):.2f} | "
          f"tỉ lệ điểm 3 {sum(x['score'] == 3 for x in results) / len(results):.0%}")
    return results